# SA-Jev · 公开 LightRAG 索引与数据准备
默认使用三套公开 N500 图，不对训练段落重新付费建图。产物保存到 MyDrive/SA-Jev/public_lightrag_v1/pilot 或 full；旧 full 目录保留。full 表示完整研究协议：8,000+4,000 道训练题、每数据集 500 道公开测试题，不表示 benchmark 全测试集。CPU 可运行，下载约 561 MB 索引和小型 MiniLM；不下载 OpenJev-4B。首次先跑 pilot；成功结束后默认刷新 Drive 并断开。

In [1]:
from pathlib import Path
import os, sys, subprocess, zipfile
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = Path('/content/drive/MyDrive/SA-Jev')
    PROJECT = Path('/content/graphrag-jev')
except ImportError:
    DRIVE = Path(os.environ.get('SAJEV_WORKDIR', '../outputs/colab_local')).resolve()
    PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DRIVE.mkdir(parents=True, exist_ok=True)
REPO_URL = os.environ.get('SAJEV_REPO_URL', 'https://github.com/BoldHu/SA-Jev.git')  # Override with SAJEV_REPO_URL if needed.
SYNC_GIT = True
if not PROJECT.exists():
    if REPO_URL:
        subprocess.run(['git', 'clone', REPO_URL, str(PROJECT)], check=True)
    elif (DRIVE / 'sa-jev-source.zip').exists():
        PROJECT.mkdir(parents=True)
        with zipfile.ZipFile(DRIVE / 'sa-jev-source.zip') as archive:
            archive.extractall(PROJECT)
    else:
        raise RuntimeError('Set SAJEV_REPO_URL or copy sa-jev-source.zip to MyDrive/SA-Jev/')
if SYNC_GIT and (PROJECT / '.git').exists():
    status = subprocess.check_output(['git', '-C', str(PROJECT), 'status', '--porcelain'], text=True)
    if status.strip():
        print('Local edits present; preserving them. Git synchronization skipped.')
    else:
        subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
os.chdir(PROJECT)
MODE = 'full'  # Use 'full' after the pilot; separate Drive directories.
AUTO_DISCONNECT = True  # Release Colab after successful preparation and Drive flush.
from uuid import uuid4
PREPARATION_RUN_ID = uuid4().hex
ROOT = DRIVE / 'public_lightrag_v1' / MODE
ROOT.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(DRIVE / 'huggingface')
PUBLIC_ASSETS = DRIVE / 'public_lightrag_assets'
CONFIG_PATH = PROJECT / 'configs/public_lightrag.yaml'
print('Code:', PROJECT, 'Artifacts:', ROOT)


Mounted at /content/drive
Code: /content/graphrag-jev Artifacts: /content/drive/MyDrive/SA-Jev/public_lightrag_v1/full


In [2]:
# Run this in each fresh Colab runtime. No model weights are downloaded here.
subprocess.run([sys.executable, 'scripts/bootstrap_upstream.py'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[train,retrieval,test]'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', 'vendor/OpenJev'], check=True)
if 'google.colab' in sys.modules:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'fla-core==0.5.2'], check=True)
# Activate editable-install .pth files in this already-running notebook kernel.
import importlib, site
site_dirs = site.getsitepackages()
if site.ENABLE_USER_SITE:
    site_dirs.append(site.getusersitepackages())
for site_dir in dict.fromkeys(site_dirs):
    site.addsitedir(site_dir)
importlib.invalidate_caches()
import sa_jev
print('SA-Jev source:', sa_jev.__file__)
# If package installation asks for a runtime restart, restart before importing torch.
from sa_jev.experiments import load_config, environment_manifest
config = load_config(CONFIG_PATH)
environment_manifest(ROOT / 'environment.json', config)
def cli(command, *args):
    from collections import deque
    tail = deque(maxlen=40)
    cmd = [sys.executable, '-u', '-m', 'sa_jev.cli', command, '--root', str(ROOT), '--config', str(CONFIG_PATH), *map(str,args)]
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                tail.append(line)
            returncode = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise
    if returncode:
        raise RuntimeError(f'sajev {command} failed (exit {returncode}). Last output: ' + ''.join(tail))


SA-Jev source: /content/graphrag-jev/src/sa_jev/__init__.py


In [3]:
# Read the same credential file in every notebook. Fill it in once.
from sa_jev.environment import load_environment
ENV_FILE = (DRIVE if 'google.colab' in sys.modules else PROJECT) / '.env'
credential_status = load_environment(ENV_FILE, template=PROJECT / '.env.example')
print('Credential file:', ENV_FILE)
for name in ('OPENAI_API_KEY', 'OPENAI_BASE_URL', 'TYPESAFE_API_KEY', 'HF_TOKEN'):
    print(name + ':', 'loaded' if credential_status[name] else 'empty')
if not credential_status['OPENAI_API_KEY']:
    print('Fill OPENAI_API_KEY in the file above, save it, then rerun this cell before building the index or answering questions.')


Credential file: /content/drive/MyDrive/SA-Jev/.env
OPENAI_API_KEY: loaded
OPENAI_BASE_URL: loaded
TYPESAFE_API_KEY: loaded
HF_TOKEN: loaded


In [4]:
import torch
from pathlib import Path
from tempfile import mkdtemp
from sa_jev.smoke import run

smoke_parent = ROOT / "smoke"
smoke_parent.mkdir(parents=True, exist_ok=True)

smoke_dir = Path(mkdtemp(prefix="run_", dir=smoke_parent))
print("本次测试目录：", smoke_dir)

summary = run(
    smoke_dir,
    device="cuda" if torch.cuda.is_available() else "cpu",
)
summary

本次测试目录： /content/drive/MyDrive/SA-Jev/public_lightrag_v1/full/smoke/run_jnwp6lzn
{'step': 1, 'epoch': 0, 'rank': 1.7917581796646118, 'suf': 0.6931471526622772, 'rel': 0.6931471526622772, 'dpo': 0.0, 'stability': 0.0, 'allocated_gib': 0.0, 'reserved_gib': 0.0, 'peak_allocated_gib': 0.0, 'peak_reserved_gib': 0.0, 'seconds': 1.00818300800006, 'eta_seconds': 1.00818300800006, 'grad_norm': 0.0001167683003586717, 'valid_rank': 1, 'valid_suf': 2, 'valid_rel': 10, 'task': 2.6235347628593444, 'total_loss': 2.6235347628593444}
{'step': 2, 'epoch': 0, 'rank': 1.6094295978546143, 'suf': 0.693147212266922, 'rel': 0.6931336522102356, 'dpo': 0.0, 'stability': 0.0, 'allocated_gib': 0.0, 'reserved_gib': 0.0, 'peak_allocated_gib': 0.0, 'peak_reserved_gib': 0.0, 'seconds': 1.9943327590001445, 'eta_seconds': 0.0, 'grad_norm': 0.0002577165432740003, 'valid_rank': 1, 'valid_suf': 2, 'valid_rel': 9, 'task': 2.4412035405635835, 'total_loss': 2.4412035405635835, 'validation': {'rank': 1.7005923390388489, 'suf'

{'test_only': True,
 'output_directory': '/content/drive/MyDrive/SA-Jev/public_lightrag_v1/full/smoke/run_jnwp6lzn',
 'model': 'tiny causal encoder + actual upstream OpenJev DecisionHead/OptionSetInteractor',
 'data': 'synthetic fixtures',
 'device': 'cpu',
 'train_states': 6,
 'validation_states': 3,
 'preference_pairs': 6,
 'stage1_optimizer_steps': 2,
 'stage2_optimizer_steps': 2,
 'checkpoint_roundtrip': True,
 'evaluation_rows': 1,
 'evaluation_errors': 0,
 'peak_reserved_gib': 0.0,
 'not_verified': ['trained OpenJev-4B weights',
  'A100 80GB peak',
  'live LightRAG extraction',
  'official Jev API',
  'scientific performance']}

公开图的向量编码器为 all-MiniLM-L6-v2（384 维、256 token），与 BGE 不兼容。本地下载固定版本，仅用于查询和兼容性检查，不调用 embedding API。源发布未记录编码器 revision，加载图后用已存向量进行兼容性校验。

In [5]:
from sa_jev.embeddings import check_embeddings
embedding_check = await check_embeddings(config['retrieval'], ROOT/'embedding_check.json')
embedding_check

Loading local embedding: sentence-transformers/all-MiniLM-L6-v2@1110a243fdf4706b3f48f1d95db1a4f5529b4d41, device=cpu


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Local embedding ready in 14.1s: dim=384, batch=16, max_length=256, normalized=True
Embedding check: (2, 384), finite=True, 0.08s, GPU peak reserved=0.000 GiB


{'backend': 'local',
 'model': 'sentence-transformers/all-MiniLM-L6-v2',
 'revision': '1110a243fdf4706b3f48f1d95db1a4f5529b4d41',
 'dimension': 384,
 'device': 'cpu',
 'load_seconds': 27.14742301199999,
 'encode_seconds': 0.08256804500001635,
 'texts_per_second': 36.33366879401596,
 'document_norms': [1.0, 1.0],
 'toy_top1_correct': True,
 'peak_gpu_reserved_gib': 0,
 'finite_vectors': True,
 'test_only': True}

In [7]:
from pathlib import Path
import importlib
import sa_jev.public_indexes as public_indexes

path = Path(public_indexes.__file__)
source = path.read_text(encoding="utf-8")

old = 'm.filename == "large_scale_lightrag/" + actual'
new = 'm.orig_filename.replace(chr(92), "/") == "large_scale_lightrag/" + actual'

if old in source:
    path.write_text(source.replace(old, new), encoding="utf-8")
elif "m.orig_filename.replace" not in source:
    raise RuntimeError("代码版本不匹配，请停止并反馈")

importlib.invalidate_caches()
importlib.reload(public_indexes)
from sa_jev.public_indexes import import_public_indexes

print("路径兼容修复完成，请重新运行刚才报错的单元。")

路径兼容修复完成，请重新运行刚才报错的单元。


下载并校验公开 ZIP，按重复条目分别导入三套图；2Wiki 使用包内完整冲突副本，详见 public_manifest.json。复用旧 full/raw 下载（若完整），否则下载原始训练源。仅采样训练问题的段落用于 BM25 难例，不调用建图 API。公开 500 题全部保留为测试；另从未用的 HippoRAG 问题族取验证与独立扰动题，不注入缺失支持证据。

In [8]:
from sa_jev.public_indexes import download_public_assets, import_public_indexes, prepare_public_data
from sa_jev.preparation import download_sources
from sa_jev.io import read_json
archive = download_public_assets(PUBLIC_ASSETS)
index_manifest = import_public_indexes(archive, ROOT)
raw = ROOT/'raw'
legacy = DRIVE/'full/raw'
if MODE == 'full' and (legacy/'download_manifest.json').exists() and read_json(legacy/'download_manifest.json').get('formal'):
    raw = legacy
    print('Reusing existing source downloads:', raw)
download_sources(raw, limit=64 if MODE=='pilot' else None)
manifest = prepare_public_data(ROOT, PUBLIC_ASSETS, raw, config, mode=MODE)
manifest

Reusing /content/drive/MyDrive/SA-Jev/public_lightrag_assets/lightrag_indexes_N500.zip
Reusing /content/drive/MyDrive/SA-Jev/public_lightrag_assets/hotpotqa_queries.json
Reusing /content/drive/MyDrive/SA-Jev/public_lightrag_assets/musique_queries.json
Reusing /content/drive/MyDrive/SA-Jev/public_lightrag_assets/2wiki_queries.json
Imported hotpotqa: {'documents': 4927, 'chunks': 4929, 'nodes': 28858, 'edges': 26330, 'relation_vectors': 26330, 'published_questions': 500}
Imported musique: {'documents': 7674, 'chunks': 7674, 'nodes': 44172, 'edges': 45613, 'relation_vectors': 45612, 'published_questions': 500}
Imported 2wiki: {'documents': 3258, 'chunks': 3260, 'nodes': 22330, 'edges': 20543, 'relation_vectors': 20543, 'published_questions': 500}
Reusing existing source downloads: /content/drive/MyDrive/SA-Jev/full/raw
[data] Reusing /content/drive/MyDrive/SA-Jev/full/raw/musique_ans_train.jsonl
[data] Reusing /content/drive/MyDrive/SA-Jev/full/raw/musique_ans_dev.jsonl
[data] Reusing /co

{'preparation_binding': {'protocol': 'public_lightrag_n500_v1',
  'mode': 'full',
  'counts': {'hotpotqa': 8000, 'musique': 4000},
  'seed': 17,
  'validation_per_dataset': 100,
  'heldout_per_dataset': 300,
  'queries': {'hotpotqa': '3ad9c0bcbf93f41d7004ca6007049c904d2605046b314a2f7ecfb379c64cba6d',
   'musique': '98ed4e21d3076532f6388d42320fb809599c63a0d8dffca8ece5e41922be6b46',
   '2wiki': '895cba294064df0c3302c76847b1fc08d99b5619f7663dfaa3b65cd780f1cac4'},
  'raw': {'hotpotqa': '9ad5804dc246b20ceccc238f4873789493184b3f1a29f56bba242feb4e452aea',
   'musique': '0ab34edc31d1db89b392c202a1a1ae84a8b6198818324ffdc10956a940ad5d6d'}},
 'protocol': 'public_lightrag_n500_v1',
 'question_fingerprint': '6f186c3c8665ad8ac151e53e796b01192e9afc671f88d3377f73cbbd4ceb1085',
 'corpus_fingerprint': '205861e860b280873c2a4e4a71564f1b775196bd3bc6e7bebf96c0160687f451',
 'questions': 14300,
 'corpus_documents': 107583,
 'splits': {'train': 12000, 'validation': 200, 'test': 1500, 'heldout': 600},
 'dataset

在导入的三个图上执行 LightRAG 原生 mix（图＋向量）召回，缓存最大 K=256，再切片得到主实验 K=32。保留已发布图，不重新抽取文档；查询关键词提取仍会调用 .env 中的 DeepSeek。完整协议有 2,300 个查询（1,500 测试、200 验证、600 独立保留），pilot 18 个。初始缓存完成后，其余 notebook 复用候选。检查 public_pool_coverage.csv；缺失支持、未知标签仍掩蔽。

In [9]:
from sa_jev.experiments import cache_pools
await cache_pools(ROOT, config)

Loading local embedding: sentence-transformers/all-MiniLM-L6-v2@1110a243fdf4706b3f48f1d95db1a4f5529b4d41, device=cpu


SafetensorError: Error while deserializing header: header too large

In [ ]:
from sa_jev.preparation import prepare_tasks
prepare_tasks(ROOT/'data', counts=config['data']['train_counts'], mine=True, dense=config['data']['dense_mining'])

训练前完整扫描公开 OpenJevData。检出的重叠保留记录并在论文报告；未检出不代表证明完全未见。此步骤不使用独立扰动测试集调参。

In [ ]:
cli('overlap')
cli('review-packets')

In [ ]:
from sa_jev.preparation import finish_preparation
completion = finish_preparation(ROOT, config, mode=MODE, run_id=PREPARATION_RUN_ID)
completion

核对实际 K 分布和有效标签数量。缺候选不复制补齐，未知相关性不硬判负例。若排序有效状态太少，核查候选注释映射/补充核验，不能把掩码简单删掉。把独立扰动草稿人工核验后保存为 perturbations_reviewed.jsonl。

结束后自动断开：只有本次运行已成功完成才执行。先刷新并卸载 Drive，再释放 Colab 运行时。Drive 刷新失败或流程失败时不主动释放；重新连接需重装环境并挂载 Drive，已保存数据保留。本地运行会跳过此步骤。

In [ ]:
from sa_jev.environment import disconnect_after_preparation
disconnect_after_preparation(ROOT, run_id=PREPARATION_RUN_ID, enabled=AUTO_DISCONNECT)